In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import metpy.calc as mpcalc
import numpy as np
import xarray as xr
import glob
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import pandas as pd
import cmocean.cm as cmo
from matplotlib.patches import Rectangle

In [2]:
# get dataset 
data = xr.open_dataset('/work/uo1075/u241321/data/ahfs_1970-2019_assi_dt.nc', decode_times=False)
data_l = xr.open_dataset('/work/uo1075/u241321/data/ahfl_1970-2019_assi_dt.nc', decode_times=False)

var = np.mean(data['__xarray_dataarray_variable__'], axis=1) + np.mean(data_l['__xarray_dataarray_variable__'], axis=1) 

variable = 'tflux'
y = np.load("/work/uo1075/u241321/data/y310_T.npy")   # T from band-pass filtering, 3-30yr  1972-2017, cut 4 years due to filtering


field = var.stack(spatial=('lat','lon')).dropna(dim="spatial") #time,space

from sklearn.linear_model import LinearRegression
def regression(x,y):

    coef = LinearRegression(fit_intercept=True).fit(x.reshape(-1, 1), y.values.reshape(-1, 1)).coef_
    

    return coef


coe = np.zeros((6, field.shape[1]))

# regression, center on 6-45 (1976-2015), 40 year (start from 0)

for m in range(0,field.shape[1],1):
        coe[0,m] = regression(y[7:47], field[2:42,m])
        coe[1,m] = regression(y[7:47], field[3:43,m])
        coe[2,m] = regression(y[7:47], field[4:44,m])
        coe[3,m] = regression(y[7:47], field[5:45,m])
        coe[4,m] = regression(y[7:47], field[6:46,m])
        coe[5,m] = regression(y[7:47], field[7:47,m])
        
coe1 = xr.DataArray(coe,  
                    dims=['mode','spatial'],
                    coords=dict(
                        spatial=field.spatial,
                         mode=np.arange(1,7,1))
                    , )
# field = var.stack(spatial=('lat','lon')).dropna(dim="spatial") #time,space
spatial = field .coords["spatial"]
mode = coe1 .coords["mode"]
reg = xr.DataArray(coe1, dims = ["mode","spatial"], coords = {"mode":mode,"spatial":spatial}).unstack()        

In [5]:
variable = 'tflux'
reg.to_netcdf("/work/uo1075/u241321/data/reg_"+variable+"_T_c2_bp310.nc") 